# FRF-Based HBM & SecondOrderODE Refactoring
**Changes since commit `cfe62d0 -> d4662e7`** — 12.05.2026

---

| File | Changes |
|---|---|
| `frequency_domain.py` | Caching in `FourierOmegaPoint`, refactored `SecondOrderODE`, new `FrequencyDomainFRF` |
| `core.py` | `freq_domain_ode` parameter, dict-copy fix |
| `__init__.py` | Export of `FrequencyDomainFRF` |

---
## 1 · Refactoring `FrequencyDomainSecondOrderODE`

### 1.1 Caching in `FourierOmegaPoint`

Previously, time-domain quantities (displacement, velocity, Jacobians) were recomputed from scratch in every method call. Each Newton iteration called `compute_nonlinear_term` and `compute_jacobian_nonlinear_term` independently — both recomputed the same $\mathbf{q}(\tau)$ and $\dot{\mathbf{q}}(\tau)$ time series.

**Fix:** Five cache attributes added to `FourierOmegaPoint.__init__`:

```python
self.time_series_derivative         = None   # qdot(τ_k)
self.second_adimensional_time_derivative = None  # -n² Q_n  (adim. acceleration)
self.Gdot                           = None   # FFT(∂f_nl/∂qdot)
self.nonlinear_term_cache           = None   # Fourier_Real of f_nl
self.Y_frf_cache                    = None   # assembled FRF matrix Y(ω)
```

Two cached methods were **moved from `FrequencyDomainSecondOrderODE_Real` into `FourierOmegaPoint`**, since they depend only on the state and $\omega$, not on the system:

**Velocity time series** (with caching):

$$\dot{\mathbf{q}}(\tau_k) = \omega \cdot \mathrm{iDFT}\{j n \, \mathbf{Q}_n\}_k$$

```python
def compute_time_series_derivative(self):
    if self.time_series_derivative is None:
        qdot_coefficients = self.omega * self.fourier.get_adimensional_time_derivative()
        qdot_fourier = Fourier_Real(qdot_coefficients)
        Fourier_Real.compute_time_series(qdot_fourier)
        self.time_series_derivative = qdot_fourier.time_series
    return self.time_series_derivative
```

**Second adimensional time derivative** (adimensional acceleration coefficients, with caching):

$$\mathbf{q}''_n = -n^2 \mathbf{Q}_n$$

```python
def compute_second_adimensional_time_derivative(self):
    if self.second_adimensional_time_derivative is None:
        self.second_adimensional_time_derivative = \
            einsum('i,ijk->ijk', Fourier.harmonics**2, self.fourier.coefficients) * -1
    return self.second_adimensional_time_derivative
```

---
### 1.2 `Fourier_Real.compute_time_series` — Guard & Return

Previously, `compute_time_series` always recomputed the iFFT, even if called twice on the same object.

```python
# Before
def compute_time_series(self) -> None:
    shape = ...
    self.time_series = irfft(...)   # always recomputes

# After
def compute_time_series(self) -> None:
    if self.time_series is None:    # guard: skip if already computed
        shape = ...
        self.time_series = irfft(...)
    return self.time_series         # return value for convenience
```

---
### 1.3 `FrequencyDomainSecondOrderODE_Real.__init__` — Pre-computed Kronecker Matrices

The linear Jacobian

$$\frac{\partial \mathbf{R}^\mathrm{lin}}{\partial \mathbf{Q}} = -\omega^2 (\mathbf{n}^2 \otimes \mathbf{M}) + j\omega (\mathbf{n} \otimes \mathbf{C}) + (\mathbf{I}_{N_h} \otimes \mathbf{K})$$

is $\omega$-dependent, but the Kronecker products $\mathbf{n}^2 \otimes \mathbf{M}$, $\mathbf{n} \otimes \mathbf{C}$, $\mathbf{I} \otimes \mathbf{K}$ are **constant** throughout the continuation. They are now computed once in `__init__`:

```python
def __init__(self, second_order_ode: SecondOrderODE):
    super().__init__(second_order_ode)
    self.kron_mass      = kron(diag(Fourier.harmonics ** 2), self.ode.mass_matrix)
    self.kron_damping   = kron(diag(Fourier.harmonics),      self.ode.damping_matrix)
    self.kron_stiffness = kron(eye(Fourier.number_of_harmonics), self.ode.stiffness_matrix)
```

---
### 1.4 Refactored `compute_residue_RI`

The residue for harmonic $n$:

$$\mathbf{R}_n = \underbrace{\left(-\omega^2 n^2 \mathbf{M} + j\omega n \mathbf{C} + \mathbf{K}\right)\mathbf{Q}_n}_{\text{linear}} + \mathbf{F}^\mathrm{nl}_n - \mathbf{F}^\mathrm{ext}_n = 0$$

The mass term uses $\omega^2 \mathbf{M} \mathbf{q}''_n = \omega^2 \mathbf{M} (-n^2 \mathbf{Q}_n)$, now via the cached method:

```python
# Before — explicit einsum every call
linear_term = (self.ode.stiffness_matrix @ state.coefficients
    + einsum('i,ijk->ijk', -n**2 * ome**2, self.ode.mass_matrix @ state.coefficients)
    + einsum('i,ijk->ijk',  1j*n*ome,      self.ode.damping_matrix @ state.coefficients))

# After — cached adimensional derivatives
linear_term = (self.ode.stiffness_matrix @ state.coefficients
    + (ome**2) * self.ode.mass_matrix @ x.compute_second_adimensional_time_derivative()
    + ome       * self.ode.damping_matrix @ state.get_adimensional_time_derivative())
```

---
### 1.5 Refactored `compute_derivative_wrt_omega_RI` — Introduction of $\dot{G}$

The derivative of the residue w.r.t. $\omega$ (holding Fourier coefficients $\mathbf{Q}$ fixed), **per harmonic $n$**:

$$\frac{\partial \mathbf{R}_n}{\partial \omega}\Bigg|_{\mathbf{Q}} = \underbrace{-2\omega n^2 \mathbf{M}\mathbf{Q}_n + jn\,\mathbf{C}(jn\mathbf{Q}_n)}_{\text{linear}} + \underbrace{\sum_{m} \dot{G}_{n-m} \cdot jm\mathbf{Q}_m}_{\text{nonlinear (convolution over harmonics)}}$$

where

$$\dot{G}_{n} = \frac{1}{N}\mathrm{DFT}\!\left\{\frac{\partial \mathbf{f}^\mathrm{nl}}{\partial \dot{\mathbf{q}}}\right\}_{n}$$

**Before** — recomputed $\partial f^\mathrm{nl}/\partial\dot{q}$ inline every call, no caching:

```python
Fourier_Real.compute_time_series(state)
q_prime_fourier = Fourier_Real(einsum('i,ijk->ijk', 1j*n, state.coefficients))
Fourier_Real.compute_time_series(q_prime_fourier)
dfnldqdot = self.ode.jacobian_nonlinear_term_qdot(q, q_prime*ome, ...)
derivative_nonlinear = Fourier_Real.new_from_time_series(dfnldqdot @ q_prime)
```

**After** — uses cached `Gdot` (shared with `compute_jacobian_nonlinear_term`):

```python
qdot_adim = state.get_adimensional_time_derivative()          # {jn Q_n} (adim., cached)
derivative_linear = (2*ome * M @ x.compute_second_adim_deriv()
                   + C @ qdot_adim)
Gdot = self.compute_Gdot(x)                                   # Ġ_n = DFT(∂f/∂qdot)/N, cached
derivative_nonlinear_RI = vstack((Gdot.RR @ qdot_R + Gdot.RI @ qdot_I,
                                  Gdot.IR @ qdot_R + Gdot.II @ qdot_I))
```

---
### 1.6 New method `compute_Gdot` — Shared Cache for $\dot{G}$

$\dot{G}$ appears in **both** `compute_jacobian_nonlinear_term` and `compute_derivative_wrt_omega_RI`. Previously it was computed separately in each — now it is computed once and stored on `x`:

$$\dot{G}_{n} = \frac{1}{N}\mathrm{DFT}\!\left\{\frac{\partial \mathbf{f}^\mathrm{nl}}{\partial \dot{\mathbf{q}}}\right\}_{n}$$

The factor $\frac{1}{N}$ comes from the DFT normalization convention used throughout the code (consistent with `JacobianFourier_Real.new_from_time_series`).

```python
def compute_Gdot(self, x: FourierOmegaPoint) -> JacobianFourier_Real:
    if x.Gdot is None:
        dfnldqdot_time_series = self.ode.jacobian_nonlinear_term_qdot(
            x.fourier.time_series,
            x.time_series_derivative,
            Fourier.adimensional_time_samples
        )
        x.Gdot = JacobianFourier_Real.new_from_time_series(dfnldqdot_time_series)
    return x.Gdot
```

---
### 1.7 Refactored `compute_jacobian_nonlinear_term`

The nonlinear Jacobian in frequency domain (chain rule for $\mathbf{f}^\mathrm{nl}(\mathbf{q}, \omega\mathbf{q}')$):

$$\frac{\partial F^\mathrm{nl}_n}{\partial Q_m} = G_{n-m} + j\omega m \, \dot{G}_{n-m}$$

In RI-block form, the velocity contribution scales the columns by $\omega \cdot \mathbf{n}$:

| Block | Before | After |
|---|---|---|
| Pre-computation | Rebuilt $\dot{\mathbf{q}}$ Fourier series each call | Uses `x.time_series_derivative` (cached) |
| $\dot{G}$ | `JacobianFourier_Real.new_from_time_series(dfnldqdot)` inline | `self.compute_Gdot(x)` (shared cache) |
| Column scaling | `kron(diag(ome * n), eye(d))` rebuilt each call | `x.omega * self.jacobian_adimensional_time_derivative_term` (pre-built in `__init__`) |

```python
def compute_jacobian_nonlinear_term(self, x):
    dfnldq = self.ode.jacobian_nonlinear_term(x.fourier.time_series,
                                              x.time_series_derivative, ...)
    G    = JacobianFourier_Real.new_from_time_series(dfnldq)
    Gdot = self.compute_Gdot(x)                          # cached
    col_scale = x.omega * self.jacobian_adimensional_time_derivative_term
    return JacobianFourier_Real(
        RR = G.RR + Gdot.RI @ col_scale,
        RI = G.RI - Gdot.RR @ col_scale,
        IR = G.IR + Gdot.II @ col_scale,
        II = G.II - Gdot.IR @ col_scale
    )
```

---






---
## 2 · New Class `FrequencyDomainFRF`

### 2.1 Motivation

The standard `FrequencyDomainSecondOrderODE` formulation requires explicit $\mathbf{M}$, $\mathbf{C}$, $\mathbf{K}$ matrices. For **experimental** substructures or **FEM models**, only the Frequency Response Function (FRF)

$$\mathbf{Y}(\omega) = \left(-\omega^2 \mathbf{M} + j\omega \mathbf{C} + \mathbf{K}\right)^{-1}$$

is available — measured or pre-computed. `FrequencyDomainFRF` enables HBM with such data as a direct replacement for `FrequencyDomainSecondOrderODE_Real`.

---
### 2.2 FRF-Based Residue

Multiply the standard residue $\mathbf{Z}(\omega)\mathbf{Q} + \mathbf{F}^\mathrm{nl} = \mathbf{F}^\mathrm{ext}$ from the left by $\mathbf{Y}(\omega) = \mathbf{Z}^{-1}$:

$$\boxed{\mathbf{R} = \mathbf{Q} + \mathbf{Y}(\omega)\,\mathbf{F}^\mathrm{nl}(\mathbf{Q}) - \mathbf{Y}(\omega)\,\mathbf{F}^\mathrm{ext} = \mathbf{0}}$$

For multiple harmonics, $\mathbf{Y}$ is assembled as a block-diagonal matrix:

$$\mathbf{Y}_{\mathrm{block}} = \mathrm{diag}\!\left[\mathbf{Y}(n_1 \omega),\; \mathbf{Y}(n_2 \omega),\; \ldots\right]$$

where each block $\mathbf{Y}(n_k \omega)$ is the $d \times d$ FRF matrix interpolated at the frequency of harmonic $n_k$.

**`compute_residue_RI` implementation:**

```python
def compute_residue_RI(self, x: FourierOmegaPoint) -> array:
    state = x.fourier
    nonlinear_term = self.compute_nonlinear_term(x)
    Q    = vstack(state.coefficients)              # (Nh*d, 1) complex
    Y    = self.get_FRF(x)                         # (Nh*d, Nh*d) complex, block-diagonal
    Fnl  = vstack(nonlinear_term.coefficients)     # (Nh*d, 1) complex
    Fext = vstack(self.external_term.coefficients) # (Nh*d, 1) complex
    R = Q + Y @ Fnl - Y @ Fext
    return vstack((R.real, R.imag))                # (2*Nh*d, 1) real
```

---
### 2.3 FRF-Based Jacobian

Differentiating $\mathbf{R} = \mathbf{Q} + \mathbf{Y}\mathbf{F}^\mathrm{nl} - \mathbf{Y}\mathbf{F}^\mathrm{ext}$ w.r.t. $\mathbf{Q}$ (with $\mathbf{Y}$, $\mathbf{F}^\mathrm{ext}$ independent of $\mathbf{Q}$):

$$\frac{\partial \mathbf{R}}{\partial \mathbf{Q}} = \mathbf{I} + \mathbf{Y}\,\frac{\partial \mathbf{F}^\mathrm{nl}}{\partial \mathbf{Q}}$$

In RI form, using $\mathbf{Y}_{\mathrm{RI}} = \begin{bmatrix} \mathrm{Re}(\mathbf{Y}) & -\mathrm{Im}(\mathbf{Y}) \\ \mathrm{Im}(\mathbf{Y}) & \mathrm{Re}(\mathbf{Y}) \end{bmatrix}$:

$$\boxed{\mathbf{J}_{\mathrm{RI}} = \mathbf{I} + \mathbf{Y}_{\mathrm{RI}}\,\mathbf{J}^\mathrm{nl}_{\mathrm{RI}}}$$

```python
def compute_jacobian_of_residue_RI(self, x: FourierOmegaPoint) -> array:
    Jnl    = self.compute_jacobian_nonlinear_term(x)
    Jnl_RI = block([[Jnl.RR, Jnl.RI], [Jnl.IR, Jnl.II]])
    Y_RI   = self.FRF_to_RI(self.get_FRF(x))
    return eye(self.real_dimension) + Y_RI @ Jnl_RI
```

---
### 2.4 Derivative w.r.t. $\omega$

Differentiating $\mathbf{R} = \mathbf{Q} + \mathbf{Y}(\omega)\mathbf{F}^\mathrm{nl} - \mathbf{Y}(\omega)\mathbf{F}^\mathrm{ext}$ w.r.t. $\omega$ at fixed $\mathbf{Q}$:

$$\frac{\partial \mathbf{R}}{\partial \omega}\Bigg|_{\mathbf{Q}} = \underbrace{\frac{d\mathbf{Y}}{d\omega}\left(\mathbf{F}^\mathrm{nl} - \mathbf{F}^\mathrm{ext}\right)}_{\text{FRF changes with }\omega} + \underbrace{\mathbf{Y}\,\frac{\partial \mathbf{F}^\mathrm{nl}}{\partial \omega}\Bigg|_{\mathbf{Q}}}_{\text{velocity-dependent nonlinearity}}$$

The second term arises because $\dot{\mathbf{q}} = \omega \mathbf{q}'$, so $\frac{\partial \mathbf{F}^\mathrm{nl}}{\partial \omega}\big|_{\mathbf{Q}} = \dot{G} \cdot \mathbf{q}'$.

$\frac{d\mathbf{Y}}{d\omega}$ is approximated via **central finite differences** (forward difference at the left boundary):

$$\frac{d\mathbf{Y}}{d\omega} \approx \frac{\mathbf{Y}(\omega+h) - \mathbf{Y}(\omega-h)}{2h}$$

```python
def compute_FRF_derivative_wrt_omega_RI(self, x):
    omega = x.omega
    if omega < self.omega_frf[0] + self.fd_step:        # left boundary: forward diff
        dY = (self.compute_FRF(omega + self.fd_step) - self.get_FRF(x)) / self.fd_step
    else:                                               # central diff
        dY = (self.compute_FRF(omega + self.fd_step)
            - self.compute_FRF(omega - self.fd_step)) / (2 * self.fd_step)
    return self.FRF_to_RI(dY)
```

---
### 2.5 FRF Interpolation

Measured FRF data is available at discrete frequency points $\omega_1, \ldots, \omega_N$. For the HBM continuation, $\mathbf{Y}(n_k \omega)$ must be evaluated at arbitrary $\omega$.

Real and imaginary parts are interpolated **independently** using cubic splines:

$$\mathbf{Y}(\omega) \approx s_\mathrm{Re}(\omega) + j\, s_\mathrm{Im}(\omega)$$

```python
self.interp_real = CubicSpline(omega_frf, Y_frf.real)
self.interp_imag = CubicSpline(omega_frf, Y_frf.imag)

def interpolate_FRF(self, omega) -> array:  # omega shape: (Nh,)
    # warns if outside measured range (extrapolation)
    return self.interp_real(omega) + 1j * self.interp_imag(omega)  # shape: (Nh, d, d)
```

The assembled block-diagonal FRF matrix:

```python
def compute_FRF(self, omega):
    omega_harmonics = Fourier.harmonics * omega          # [n1*ω, n2*ω, ...]
    Y_blocks = self.interpolate_FRF(omega_harmonics)     # (Nh, d, d)
    Y = zeros((Nh*d, Nh*d), dtype=complex)
    for k in range(Nh):
        Y[k*d:(k+1)*d, k*d:(k+1)*d] = Y_blocks[k]       # block diagonal
    return Y
```

---
### 2.6 `get_FRF` — Caching the Assembled FRF

Assembling $\mathbf{Y}_{\mathrm{block}}$ involves Nh spline evaluations. Since $\omega$ is fixed within one Newton iteration, the result is cached on `x`:

```python
def get_FRF(self, x: FourierOmegaPoint):
    if x.Y_frf_cache is None:
        x.Y_frf_cache = self.compute_FRF(x.omega)
    return x.Y_frf_cache
```

---
### 2.7 Inheritance Diagram

```
FrequencyDomainSecondOrderODE          (abstract base: residue, dR/dω)
    └── FrequencyDomainSecondOrderODE_Real    (AFT, Jacobian, Gdot, kron matrices)
            └── FrequencyDomainFRF            (FRF-based residue & Jacobian,
                                               spline interpolation, finite diff. dY/dω)
```

`FrequencyDomainFRF` **overrides** three methods from `FrequencyDomainSecondOrderODE`:

| Method | Parent computes | FRF version computes |
|---|---|---|
| `compute_residue_RI` | $\mathbf{Z}(\omega)\mathbf{Q} + \mathbf{F}^\mathrm{nl} - \mathbf{F}^\mathrm{ext}$ | $\mathbf{Q} + \mathbf{Y}(\omega)(\mathbf{F}^\mathrm{nl} - \mathbf{F}^\mathrm{ext})$ |
| `compute_jacobian_of_residue_RI` | $\mathbf{J}_\mathrm{lin} + \mathbf{J}_\mathrm{nl}$ | $\mathbf{I} + \mathbf{Y}_{\mathrm{RI}}\,\mathbf{J}^\mathrm{nl}_{\mathrm{RI}}$ |
| `compute_derivative_wrt_omega_RI` | $2\omega\mathbf{M}\mathbf{q}'' + \mathbf{C}\mathbf{q}' + \dot{G}\mathbf{q}'$ | $\frac{d\mathbf{Y}}{d\omega}(\mathbf{F}^\mathrm{nl} - \mathbf{F}^\mathrm{ext}) + \mathbf{Y}\dot{G}\mathbf{q}'$ |

`compute_nonlinear_term`, `compute_jacobian_nonlinear_term`, `compute_Gdot` are **inherited unchanged**.

---
## 3 · Changes in `core.py`

### 3.1 `freq_domain_ode` parameter

`HarmonicBalanceMethod` now accepts a pre-built frequency domain object directly — needed for `FrequencyDomainFRF` which has a different constructor signature than `SecondOrderODE`:

```python
# Before: only first_order_ode and second_order_ode
ode = first_order_ode if first_order_ode is not None else second_order_ode

# After: three-way fallback
ode = first_order_ode  if first_order_ode  is not None else \
      second_order_ode if second_order_ode is not None else \
      freq_domain_ode.ode

if freq_domain_ode is not None:
    self.freq_domain_ode = freq_domain_ode          # use as-is
elif second_order_ode is not None:
    self.freq_domain_ode = FrequencyDomainSecondOrderODE_Real(second_order_ode)
...
```

### 3.2 Dict copy for `solver_kwargs`

```python
# Before — mutated the caller's dict!
solver_kwargs["absolute_tolerance"] *= sqrt(2) / Fourier.number_of_time_samples

# After — safe copy first
solver_kwargs = dict(solver_kwargs)
solver_kwargs["absolute_tolerance"] *= sqrt(2) / Fourier.number_of_time_samples
```